In [1]:
%pip install bert_score sentence_transformers osmnx


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import pickle
import numpy as np
import osmnx as ox
import networkx as nx
import spacy
import torch
import re
import requests
from tqdm import tqdm
from gliner import GLiNER
from sentence_transformers import SentenceTransformer
from eval.evaluation import prompt_based_route_evaluator
from routing.synset import OSMSemanticBridge
from scipy.special import softmax

# --- 1. GLiNER + Llama MANAGER (Judge & Extractor) ---

class ModelManager:
    def __init__(self, model_id="urchade/gliner_medium-v2.1", llama_model="llama3.2:3b"):
        print(f"Initializing Hybrid Manager (GLiNER + Llama)...")
        self.model = GLiNER.from_pretrained(model_id)
        self.labels = ["avoid_feature", "preferred_feature"]
        self.llama_model = llama_model
        self.ollama_url = "http://localhost:11434/api/generate"

    def extract_intent(self, user_prompt, tag_schema):
        # Keep the existing extraction logic as it is working well
        entities = self.model.predict_entities(user_prompt, self.labels, threshold=0.55)
        intent = {"avoid_tags": {}, "prefer_tags": {}}
        for ent in entities:
            text = ent["text"].lower()
            label = "avoid_tags" if ent["label"] == "avoid_feature" else "prefer_tags"
            for cat, values in tag_schema["discrete"].items():
                for val in values:
                    if val in text or text in val:
                        if cat not in intent[label]:
                            intent[label][cat] = set()
                        intent[label][cat].add(val)
        return {k: {cat: list(vals) for cat, vals in v.items()} for k, v in intent.items()}

    def judge_candidates(self, user_prompt, candidates, G, baseline_dist):
        if not candidates or len(candidates) <= 1:
            return candidates[0] if candidates else None

        # Step 1: Extract symbolic intent using the full schema
        schema_keys = ['highway', 'access', 'bridge', 'junction', 'ref']
        intent_data = self.extract_intent(user_prompt, {"discrete": {k: [] for k in schema_keys}}) 

        # Step 2: Build isolated candidate summaries for the LLM
        route_summaries = []
        for i, path in enumerate(candidates):
            dist = 0
            path_tags = {k: set() for k in schema_keys}
            for u, v in zip(path[:-1], path[1:]):
                edge_data = G.get_edge_data(u, v)[0]
                dist += edge_data.get('length', 0)
                for k in schema_keys:
                    val = edge_data.get(k)
                    if val:
                        if isinstance(val, list): path_tags[k].update(val)
                        else: path_tags[k].add(val)

            tag_list = [f"{k}: {list(v)}" for k, v in path_tags.items() if v]
            # Keep distance in km for easier model reasoning
            route_summaries.append(f"ID {i} | Distance: {dist/1000:.2f}km | Tags: {', '.join(tag_list)}")

        # Step 3: Create the Context-Aware Prompt
        # We include the original prompt to allow the model to interpret nuance 
        # that might have been missed in the discrete tag extraction.
        llm_prompt = (
            f"USER: {user_prompt}\n"
            f"AVOID: {json.dumps(intent_data['avoid_tags'])}\n"
            f"PREFER: {json.dumps(intent_data['prefer_tags'])}\n\n"
            "CANDIDATES:\n" + "\n".join(route_summaries) + "\n\n"
            "TASK: Return the ID of the best candidate that balances maximizing constraint satisfaction with minimizing distance in JSON format." #that balances constraints and distance
        )

        try:
            # Step 4: Constrained Decoding with Ollama JSON mode
            # Adding a 'reasoning' field helps the model process logic before picking an index.
            response = requests.post(self.ollama_url, json={
                "model": self.llama_model, 
                "prompt": llm_prompt, 
                "stream": False,
                "format": {
                    "type": "object",
                    "properties": {
                        "selected_index": { "type": "integer" }
                    },
                    "required": ["selected_index"]
                },
                "options": {
                    "temperature": 0,
                    "num_predict": 20 # Increased to accommodate the reasoning string
                } 
            })
            
            # Safe parsing of the JSON response
            raw_result = response.json().get("response", "{}")
            data = json.loads(raw_result)
            
            idx = data.get("selected_index", 0)
            reason = data.get("reasoning", "No reason provided.")
            
            # Log reasoning to tqdm to monitor for hallucinations or logic errors
            tqdm.write(f"Judge Logic: {reason}")
            tqdm.write(f"LLM Choice: {idx} (Schema Validated)")
            
            # Ensure we don't index out of bounds if the LLM hallucinates an ID
            selected_idx = max(0, min(idx, len(candidates) - 1))
            return candidates[selected_idx]
            
        except Exception as e:
            tqdm.write(f"LLM Judge Error: {e}")
            # Default to the first candidate (usually the standard SMC shortest path)
            return candidates[0]

# --- 2. SMC GENERATOR (Remains focused on particle filtering) ---

class SMCRouteGenerator:
    def __init__(self, G, bridge, st_model, n_particles=50):
        self.G = G
        self.bridge = bridge
        self.st_model = st_model
        self.n_particles = n_particles
        self.intent_cache = {}
            
    def _get_distance(self, node, target):
        n1 = self.G.nodes[node]
        n2 = self.G.nodes[target]
        return ox.distance.great_circle(n1['y'], n1['x'], n2['y'], n2['x'])

    def _calculate_edge_weight(self, edge_data, intent):
        multiplier = 1.0
        for cat in self.bridge.categories:
            val = edge_data.get(cat)
            if not val: continue
            val = val[0] if isinstance(val, list) else val
            
            # Accessing list from JSON-ified intent
            if val in intent["avoid_tags"].get(cat, []):
                multiplier *= 0.1 
            if val in intent["prefer_tags"].get(cat, []):
                multiplier *= 10.0
        return multiplier

    def find_routes(self, start_node, end_node, prompt, intent, n_candidates=3, max_steps=500):
        try:
            anchor_path = nx.astar_path(self.G, start_node, end_node, weight='length')
            anchor_nodes = set(anchor_path)
        except nx.NetworkXNoPath:
            return []

        particles = [[start_node, [start_node], 1.0, False] for _ in range(self.n_particles)]
        unique_completed_paths = {}

        for step in range(max_steps):
            active = [i for i, p in enumerate(particles) if not p[3]]
            if not active or len(unique_completed_paths) >= n_candidates:
                break

            for i in active:
                curr_node = particles[i][0]
                neighbors = list(self.G.neighbors(curr_node))
                if not neighbors:
                    particles[i][2] = 0
                    continue
                
                neigh_coords = np.array([[self.G.nodes[n]['x'], self.G.nodes[n]['y']] for n in neighbors])
                target_coord = np.array([self.G.nodes[end_node]['x'], self.G.nodes[end_node]['y']])
                
                dists_deg = np.sqrt(np.sum((neigh_coords - target_coord)**2, axis=1))
                dists = dists_deg * 111000

                on_anchor = np.array([50.0 if n in anchor_nodes else 0.0 for n in neighbors])
                
                scores = -(dists / 100.0) + on_anchor
                probs = softmax(scores/2.0)
                
                next_node = np.random.choice(neighbors, p=probs)
                edge_data = self.G.get_edge_data(curr_node, next_node)
                data = list(edge_data.values())[0] if isinstance(edge_data, dict) else edge_data
                
                particles[i][2] *= self._calculate_edge_weight(data, intent)
                particles[i][0] = next_node
                particles[i][1].append(next_node)
                
                if next_node == end_node:
                    particles[i][3] = True
                    path_tuple = tuple(particles[i][1])
                    unique_completed_paths[path_tuple] = particles[i][2]

            # Resampling
            if step % 20 == 0:
                weights = np.array([p[2] for p in particles])
                if weights.sum() > 0:
                    weights /= weights.sum()
                    idx = np.random.choice(range(self.n_particles), size=self.n_particles, p=weights)
                    particles = [[particles[j][0], list(particles[j][1]), 1.0, particles[j][3]] for j in idx]

        sorted_paths = sorted(unique_completed_paths.items(), key=lambda x: x[1], reverse=True)
        return [list(path) for path, weight in sorted_paths[:n_candidates]]

# --- 3. MAIN EXPERIMENT SCRIPT ---

if __name__ == "__main__":
    # Load Data (Assuming paths are correct for your environment)
    with open("research/pioneer_valley_v2.pkl", "rb") as f:
        G = pickle.load(f)
    with open("research/synthetic_dataset.jsonl", "r") as f:
        data = [json.loads(line) for line in f]

    tag_schema = {
        "continuous": {
            "maxspeed_imputed": {"min": 10, "max": 65, "unit": "mph"},
            "lanes": {"min": 1, "max": 6},
            "length": {"min": 2, "max": 6845}
        },
        "discrete": {
            "highway": ['residential', 'trunk', 'secondary', 'tertiary', 'primary', 'motorway_link', 
                        'unclassified', 'secondary_link', 'motorway', 'tertiary_link', 'primary_link', 'trunk_link'],
            "access": ["yes", "no"],
            "bridge": ["yes", "viaduct"],
            "junction": ["roundabout", "jughandle"],
            "ref": ['US 20', 'US 5', 'MA 9', 'MA 10', 'MA 66', 'MA 187', 'US 202', 'I 91', 'I 90', 'MA 116']
        }
    }

    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    bridge = OSMSemanticBridge(tag_schema, st_model, threshold=0.80)
    
    gliner_manager = ModelManager()
    smc_generator = SMCRouteGenerator(G, bridge, st_model, n_particles=100)
    
    gen_routes, gen_prompts = [], []

    print("\n--- Running Neurosymbolic Experiment (Mode: SMC + GLiNER Extraction + Llama Judge) ---")
    
    for idx, item in enumerate(tqdm(data[100:200])):
        try:
            s_pt = ox.geocoder.geocode(f"{item['start']}, MA, USA")
            e_pt = ox.geocoder.geocode(f"{item['end']}, MA, USA")
            s_node = ox.distance.nearest_nodes(G, X=s_pt[1], Y=s_pt[0])
            e_node = ox.distance.nearest_nodes(G, X=e_pt[1], Y=e_pt[0])
            
            if not nx.has_path(G, s_node, e_node): continue

            # 1. GLiNER Intent Extraction
            intent = gliner_manager.extract_intent(item['prompt'], tag_schema)
            
            # 2. SMC Candidate Generation
            candidates = smc_generator.find_routes(s_node, e_node, item['prompt'], intent)
            
            if candidates:
                anchor_path = nx.astar_path(G, s_node, e_node, weight='length')
                base_dist = sum(G.get_edge_data(u,v)[0].get('length',0) for u,v in zip(anchor_path[:-1], anchor_path[1:]))

                # 3. GLiNER Selection
                best_route = gliner_manager.judge_candidates(item['prompt'], candidates, G, base_dist)
                
                gen_routes.append(best_route) 
                gen_prompts.append(item['prompt'])
                
        except Exception as e:
            continue

    if gen_routes:
        evaluator = prompt_based_route_evaluator(G, gen_prompts, gen_routes, bridge)
        results = evaluator.evaluate_method()
        print("\n--- Final Consolidated Metrics ---")
        for metric, value in results.items():
            print(f"{metric:30}: {value:.4f}")
    else:
        print("No routes generated")

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bridge: Building Schema-Grounded Index for categories: ['highway', 'access', 'bridge', 'junction', 'ref']
Initializing Hybrid Manager (GLiNER + Llama)...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Final Consolidated Metrics ---
avg_path_validity             : 1.0000
avg_deviation_penalty         : 1.9722
avg_constraint_satisfaction   : 0.4669
avg_semantic_alignment_f1     : 0.7947
